## This notebook applies SSFs to sea lion data. 
### Notes
    - Colonies: 
       - 'Montague Island'
       - 'Pearson Island'
       - 'Kangaroo Island',
       - 'Liguanea Island'
       - 'Greenly Island'
       - 'South Nepture Island'
       - 'Seal Bay - Kangaroo Island'
       - 'Seal Slide - Kangaroo Island',
       - 'Little Hummock Island'
       - 'Little Wiers'
       - 'Lewis Island',
       -'Cape du Couedic'
       - 'West Waldegrave Island'
       - 'Prise Island',
       - 'Nicolas Baudin Island'
       - 'Rocky South Island',
       - 'Rocky North Island'
### - ENSO events: 
    - El Nino strong event: early 2015- May 2016
    - La Nina strong event: mid 2010 - mid 2012
### - MHW events
    - Summer 2013
    - 2015 - 2016 driven by El nino


In [2]:
### Import packages

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import cartopy.crs as ccrs
import cartopy.feature as cf
import xarray as xr
import cmocean


from matplotlib.colors import Normalize
from matplotlib.colors import ListedColormap
from matplotlib.cm import ScalarMappable
from datetime import datetime


In [3]:
### Load csv that contains sea lion data 

In [4]:
# Replace 'your_file.csv' with the actual file path
file_path = r'C:\Users\nribeiro\OneDrive - University of Tasmania\IMOS Shared Docs\CAPSTAN 2025\Datasets\sea_lion_SA.csv'

#I had to add this line because I was getting an error while importing that this column had mixed types. 
dtype_options = {'device_wmo_ref': 'object'}


# Load the CSV file into a DataFrame
df = pd.read_csv(file_path, dtype=dtype_options)
# Display the DataFrame
df.head()


,FID,measurement_id,profile_id,file_id,sattag_program,device_id,device_wmo_ref,ptt,tag_type,common_name,...,lon,lat,pressure,temp_vals,sal_vals,sal_corrected_vals,fluoro_vals,cond_vals,geom,colour
0,aatams_sattag_dm_profile_data.fid--3dc5c2da_19...,98133430,6280425,22,ct101,ct101-790-13,Q9900594,129856,SMRU CTD tag,Australian Fur Seal,...,142.7240,-38.8914,NaN,NaN,35.747,35.747,NaN,NaN,POINT (142.723999453627 -38.8913879251468),#1C0EE2
1,aatams_sattag_dm_profile_data.fid--3dc5c2da_19...,98133449,6280426,22,ct101,ct101-790-13,Q9900594,129856,SMRU CTD tag,Australian Fur Seal,...,142.6852,-38.8729,4.0,15.1700,NaN,NaN,NaN,NaN,POINT (142.685152301598 -38.8728658653189),#1C0EE2
2,aatams_sattag_dm_profile_data.fid--3dc5c2da_19...,98133450,6280426,22,ct101,ct101-790-13,Q9900594,129856,SMRU CTD tag,Australian Fur Seal,...,142.6852,-38.8729,8.0,15.1569,NaN,NaN,NaN,NaN,POINT (142.685152301598 -38.8728658653189),#1C0EE2
3,aatams_sattag_dm_profile_data.fid--3dc5c2da_19...,98133451,6280426,22,ct101,ct101-790-13,Q9900594,129856,SMRU CTD tag,Australian Fur Seal,...,142.6852,-38.8729,12.0,15.1489,NaN,NaN,NaN,NaN,POINT (142.685152301598 -38.8728658653189),#1C0EE2
4,aatams_sattag_dm_profile_data.fid--3dc5c2da_19...,98133452,6280426,22,ct101,ct101-790-13,Q9900594,129856,SMRU CTD tag,Australian Fur Seal,...,142.6852,-38.8729,16.0,15.1449,NaN,NaN,NaN,NaN,POINT (142.685152301598 -38.8728658653189),#1C0EE2


In [5]:
### I want to identify which colonies I have

print(df.columns.tolist())

['FID', 'measurement_id', 'profile_id', 'file_id', 'sattag_program', 'device_id', 'device_wmo_ref', 'ptt', 'tag_type', 'common_name', 'release_site', 'state_country', 'age_class', 'sex', 'timestamp', 'lon', 'lat', 'pressure', 'temp_vals', 'sal_vals', 'sal_corrected_vals', 'fluoro_vals', 'cond_vals', 'geom', 'colour']


In [6]:
df["release_site"].unique()

array(['Montague Island', 'Pearson Island', 'Kangaroo Island',
       'Liguanea Island', 'Greenly Island', 'South Nepture Island',
       'Seal Bay - Kangaroo Island', 'Seal Slide - Kangaroo Island',
       'Little Hummock Island', 'Little Wiers', 'Lewis Island',
       'Cape du Couedic', 'West Waldegrave Island', 'Prise Island',
       'Nicolas Baudin Island', 'Rocky South Island',
       'Rocky North Island', nan], dtype=object)

### Here is probably a good spot to make a map with the colony sites identified. 


### Standardise the names between Python and R

In [14]:
df = df.rename(columns={
    "device_id": "animal_id",
    "timestamp": "timestamp",
    "lat": "lat",
    "lon": "lon"
})

In [15]:
print(df.columns.tolist())

['FID', 'measurement_id', 'profile_id', 'file_id', 'sattag_program', 'animal_id', 'device_wmo_ref', 'ptt', 'tag_type', 'common_name', 'release_site', 'state_country', 'age_class', 'sex', 'timestamp', 'lon', 'lat', 'pressure', 'temp_vals', 'sal_vals', 'sal_corrected_vals', 'fluoro_vals', 'cond_vals', 'geom', 'colour', 'colony_group']


### Convert time

In [16]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

### Sorting into correct movement order 

In [19]:
df = df.sort_values(["animal_id", "timestamp"])

In [21]:
df[["animal_id", "timestamp", "lat", "lon"]]

,animal_id,timestamp,lat,lon
0,ct101-790-13,2013-10-27 01:16:00+00:00,-38.8914,142.7240
1,ct101-790-13,2013-10-27 01:46:00+00:00,-38.8729,142.6852
2,ct101-790-13,2013-10-27 01:46:00+00:00,-38.8729,142.6852
3,ct101-790-13,2013-10-27 01:46:00+00:00,-38.8729,142.6852
4,ct101-790-13,2013-10-27 01:46:00+00:00,-38.8729,142.6852
...,...,...,...,...
369391,ft19-633_rebat2-14,2015-02-12 18:44:00+00:00,-35.9871,137.6026
369392,ft19-633_rebat2-14,2015-02-12 18:44:00+00:00,-35.9871,137.6026
369393,ft19-633_rebat2-14,2015-02-12 18:44:00+00:00,-35.9871,137.6026
369394,ft19-633_rebat2-14,2015-02-12 18:44:00+00:00,-35.9871,137.6026


In [22]:
df["animal_id"].nunique()

46

### Convert to a proper spatial dataset
#### SSF cannot be done in latitude/longitude because distances are wrong on a sphere. We need metres.

### Create GEO Data frame
#### What this means:
    - You now have spatial points
    - Currently in WGS84 (lat/lon)

In [24]:
import geopandas as gpd

gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),
    crs="EPSG:4326"
)

#### Reproject into meters
    Australia's best std choice: "EPSG:3577"
    This gives you:
    - x, y in metres
    - correct distances
    - valid step lengths

In [26]:
gdf = gdf.to_crs("EPSG:3577")

In [27]:
df["x"] = gdf.geometry.x
df["y"] = gdf.geometry.y

### Do my movement-ready table to export to R


In [28]:
ssf_df = df[[
    "animal_id",
    "timestamp",
    "x",
    "y",
    "release_site"   # keep it, but optional
]]

#### Export to R

In [30]:
ssf_df.to_csv("sea_lion_ssf_ready.csv", index=False)

In [31]:
pwd

'C:\\Users\\nribeiro\\OneDrive - University of Tasmania\\IMOS Shared Docs\\CAPSTAN 2025\\CAPSTAN_HAB_work\\Notebooks'